# Solutions · Chapter 05-06 · Gradient descent from scratch

Worked answers to every exercise in `notebooks/05_regression/05-06_gradient_descent.ipynb`.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, SGDRegressor

warnings.filterwarnings("ignore")
np.seterr(all="ignore")

# SYNTHETIC: the chapter's 500 houses.
# TRUTH: price = 25 + 2.4 x area + 11.0 x rooms - 0.8 x age + noise(sd 18).
rng = np.random.default_rng(21)
n_houses = 500
area_m2 = rng.uniform(40, 250, n_houses)
rooms = rng.integers(1, 7, n_houses).astype(float)
age_years = rng.uniform(0, 90, n_houses)
price = (25 + 2.4 * area_m2 + 11.0 * rooms - 0.8 * age_years
         + rng.normal(0, 18, n_houses))

raw_features = np.column_stack([area_m2, rooms, age_years])
design = np.column_stack([np.ones(n_houses), raw_features])
standardised = np.column_stack([
    np.ones(n_houses),
    (raw_features - raw_features.mean(axis=0)) / raw_features.std(axis=0)])

# the chapter's ravine: a near-copy of area, drawn in the same order
almost_area = area_m2 + rng.normal(0, 3.0, n_houses)
ravine_raw = np.column_stack([area_m2, rooms, age_years, almost_area])
ravine = np.column_stack([np.ones(n_houses),
                          (ravine_raw - ravine_raw.mean(axis=0)) / ravine_raw.std(axis=0)])


def loss_of(weights, X, y):
    return float(((X @ weights - y) ** 2).mean())


def gradient_of(weights, X, y):
    return 2.0 / len(y) * X.T @ (X @ weights - y)


best_standardised = np.linalg.lstsq(standardised, price, rcond=None)[0]
optimal_loss = loss_of(best_standardised, standardised, price)
print("optimal loss %.6f" % optimal_loss)

## Quick understanding

### E1 · The gradient

$$\nabla L(w) = \frac{2}{n}X^{\top}\left(Xw - y\right)$$

`X.T @ residual` asks, **for each feature, how strongly does this feature line up with the errors the
model is still making?** A feature that is large exactly where the model under-predicts gets a large
positive entry, and its weight is pushed up.

When every entry is zero, no feature has any remaining relationship with the residuals - which is the
same statement as 05-05's `corr(residual, fitted) = 0`, and it is why that plot has a flat reference.

### E2 · The largest legal learning rate

$$\text{rate} < \frac{2}{\lambda_{\max}(H)} = \frac{1}{\lambda_{\max}\!\left(\tfrac{1}{n}X^{\top}X\right)}$$

It is in terms of the **curvature of the loss in its steepest direction**, which for squared error is the
largest eigenvalue of `X.T @ X / n` and does not depend on `y` or on where you are standing. The bound is
exact rather than a heuristic.

### E3 · Why the gradient norm is the better stopping test

**Because the loss change confuses "arrived" with "moving slowly".** A tiny learning rate produces tiny
changes in the loss at every point, including points nowhere near the minimum, so a tolerance on the loss
change accepts them as converged - the chapter's demo reported CONVERGED at a loss 559 times too high.

The gradient norm is zero **only** at a stationary point. It needs no reference value, does not care how
large your steps were, and is one line.

## Hand calculation

### E4 · Two steps by hand

Data `x = [1, 2]`, `y = [3, 5]`, model `y = wx`, no intercept.

$$L(w) = \tfrac{1}{2}\left[(w-3)^2 + (2w-5)^2\right], \qquad
L'(w) = \tfrac{2}{2}\left[1\cdot(w-3) + 2\cdot(2w-5)\right] = 5w - 13$$

**Step 1** from `w = 0`: gradient `5(0) - 13 = -13`, so `w = 0 - 0.1(-13) = ` **1.3**.

**Step 2** from `w = 1.3`: gradient `5(1.3) - 13 = -6.5`, so `w = 1.3 - 0.1(-6.5) = ` **1.95**.

The update rule collapses to `w_next = 0.5 w + 1.3`, which halves the remaining distance every step.

### E5 · The exact answer and the stability bound

The minimum is where `5w - 13 = 0`, so **w = 2.6**.

Curvature: `2/n * sum(x²) = (2/2)(1 + 4) = ` **5**.

Largest stable rate: `2 / 5 = ` **0.4**. A rate of 0.1 is a quarter of that, comfortably inside, which is
why each step moved halfway to the answer rather than overshooting.

Worth noticing: at rate `1/5 = 0.2` the multiplier `1 - 0.2(5)` is **0**, so it would land exactly on 2.6
in **one step**. That is Newton's method from 03-08, arrived at from the other direction - for a
quadratic, the perfect learning rate is one over the curvature.

### E6 · Eigenvalues 0.5 and 50

- **Condition number:** `50 / 0.5 = ` **100**.
- **Largest stable rate:** `1 / 50 = ` **0.02**.
- **After standardising to 1 and 1:** condition 1, largest rate 1, and the step count falls by roughly
  the factor the condition number fell by - **about 100 times fewer steps.**

The rate is set by the *largest* eigenvalue and the *slowness* by the smallest, which is exactly why
their ratio is the quantity that matters.

### E7 · Updates per epoch

10,000 rows at batch 200: `10,000 / 200 = ` **50 updates per epoch.**

SGD makes one update per row, so it reaches 50 updates after **50 rows - one two-hundredth of an epoch.**

**That ratio is the whole argument for SGD**, and it is also the trap the chapter's comparison exposes.
Counting updates makes SGD look 200 times more productive per epoch; counting epochs shows that each of
those updates is 200 times less informed. Neither number is wrong, and only the second is a fair
comparison.

## Coding

### E8 · A reusable gradient check

In [ ]:
def check_gradient(loss_fn, grad_fn, weights, h=1e-5):
    analytic = grad_fn(weights)
    numeric = np.zeros_like(weights, dtype=float)
    for j in range(len(weights)):
        step = np.zeros_like(weights, dtype=float)
        step[j] = h
        numeric[j] = (loss_fn(weights + step) - loss_fn(weights - step)) / (2 * h)
    scale = max(np.abs(analytic).max(), np.abs(numeric).max(), 1e-12)
    return float(np.abs(analytic - numeric).max() / scale)


probe = np.array([1.0, -2.0, 0.5, 3.0])
loss_here = lambda w: loss_of(w, standardised, price)

correct = lambda w: gradient_of(w, standardised, price)
missing_two = lambda w: gradient_of(w, standardised, price) / 2
missing_n = lambda w: 2.0 * standardised.T @ (standardised @ w - price)      # forgot the /n
wrong_sign = lambda w: -gradient_of(w, standardised, price)

for label, candidate in [("correct", correct), ("missing the 2", missing_two),
                         ("missing the 1/n", missing_n), ("sign flipped", wrong_sign)]:
    score = check_gradient(loss_here, candidate, probe)
    print("%-18s relative difference %12.3e   %s"
          % (label, score, "PASS" if score < 1e-6 else "FAIL"))

**The correct gradient scores 2.1e-09; every wrong one scores 0.5 or more.**

There is no ambiguous middle. A gradient is either right to eight or nine digits or wrong by a factor,
and the reason is that the only error left in a correct implementation is the truncation error of the
finite difference itself.

**Two practical notes.** Dividing by the larger of the two magnitudes makes the check scale-free, so the
same threshold works whatever the loss is measured in. And `h = 1e-5` is not arbitrary: much larger and
the difference approximation is inaccurate, much smaller and floating-point cancellation destroys it.
Anywhere in 1e-4 to 1e-6 is fine.

### E9 · Where convergence actually stops

In [ ]:
limit = 1 / np.linalg.eigvalsh(standardised.T @ standardised / n_houses)[-1]
print("the predicted boundary is %.6f\n" % limit)


def steps_to_converge(X, y, rate, cap=300_000):
    target = loss_of(np.linalg.lstsq(X, y, rcond=None)[0], X, y)
    weights = np.zeros(X.shape[1])
    for step in range(1, cap + 1):
        weights = weights - rate * gradient_of(weights, X, y)
        current = loss_of(weights, X, y)
        if not np.isfinite(current):
            return None
        if current - target < 1e-6:
            return step
    return None


sweep = []
for fraction in [0.5, 0.8, 0.95, 0.99, 1.0, 1.01, 1.05, 1.2]:
    count = steps_to_converge(standardised, price, fraction * limit)
    sweep.append({"fraction of the limit": fraction, "learning rate": fraction * limit,
                  "steps": count if count else np.nan})
print(pd.DataFrame(sweep).to_string(index=False, na_rep="never"))

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 4.4))
table = pd.DataFrame(sweep).dropna()
ax.plot(table["fraction of the limit"], table["steps"], "o-", color="#0072B2",
        linewidth=2.2, markersize=9)
ax.axvline(1.0, color="#D55E00", linewidth=2.2, linestyle="--",
           label="the predicted boundary")
ax.fill_betweenx([1, 1e4], 1.0, 1.3, color="#D55E00", alpha=0.12)
ax.text(1.09, 12, "never converges", color="#D55E00", fontsize=10, fontweight="bold")
ax.set_yscale("log")
ax.set_xlim(0.4, 1.3)
ax.set_ylim(1, 3000)
ax.set_xlabel("learning rate, as a fraction of 1 / lambda_max")
ax.set_ylabel("steps to converge (log scale)")
ax.set_title("The bound is exact, and the approach to it is expensive", fontsize=11.5)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

**The boundary is exactly where the formula said, and it is a cliff.**

Every rate below it converges; every rate at or above it never does, including `1.00 x limit` itself -
the inequality is strict.

**The more useful finding is the shape of the curve below the boundary.** Steps go 5, 22, 104, 538 at
0.50, 0.80, 0.95 and 0.99 of the limit. **Half the limit is a hundred times faster than 99% of it.**

So "use the largest learning rate that does not diverge" is bad advice twice over: it is fragile, because
the boundary is a cliff, and it is *slow*, because the region just under the boundary is where the
zig-zagging is worst. On this problem the sweet spot is around half.

### E10 · A learning rate that adapts

In [ ]:
def halving_descent(X, y, initial_rate, cap=60_000):
    target = loss_of(np.linalg.lstsq(X, y, rcond=None)[0], X, y)
    weights = np.zeros(X.shape[1])
    rate, previous, halvings = initial_rate, loss_of(np.zeros(X.shape[1]), X, y), 0
    for step in range(1, cap + 1):
        proposal = weights - rate * gradient_of(weights, X, y)
        current = loss_of(proposal, X, y)
        if not np.isfinite(current) or current > previous:
            rate, halvings = rate / 2, halvings + 1
            continue
        weights, previous = proposal, current
        if current - target < 1e-6:
            return step, rate, halvings
    return None, rate, halvings


ravine_limit = 1 / np.linalg.eigvalsh(ravine.T @ ravine / n_houses)[-1]
print("the ravine's stability limit is %.6f\n" % ravine_limit)

for start in [8.0, 1.0, 0.4, 0.1]:
    steps, final_rate, halvings = halving_descent(ravine, price, start)
    print("halve-on-increase from %-5g -> %8s steps, ended at rate %.6f after %d halvings"
          % (start, steps if steps else "never", final_rate, halvings))

print()
for rate in [0.49, 0.4, 0.1]:
    print("fixed rate %-5g -> %s steps" % (rate, steps_to_converge(ravine, price, rate, 60_000)))

**The halving schedule is worse than every fixed rate it was supposed to replace, and the reason is
worth understanding.**

- **Started below the limit (0.4, 0.1) it never fires.** On a convex quadratic the loss cannot increase
  at a stable rate, so the trigger condition never occurs and the schedule is just a fixed rate with
  extra code.
- **Started above the limit (8.0, 1.0) it halves down to exactly 0.5** - and the limit is 0.500017, so it
  parks itself a hundred-thousandth *under* the boundary, which E9 just showed is the slowest possible
  place to be. Neither run converged in 60,000 steps.

**Halving finds a stable rate. It has no reason to find a good one**, because the rate it lands on is
whatever power of two the starting value happens to reach.

A schedule that also grows the rate when things are going well - the "bold driver" - fixes this.

In [ ]:
def bold_driver(X, y, initial_rate, grow=1.05, shrink=0.5, cap=60_000):
    target = loss_of(np.linalg.lstsq(X, y, rcond=None)[0], X, y)
    weights = np.zeros(X.shape[1])
    rate, previous, used = initial_rate, loss_of(np.zeros(X.shape[1]), X, y), []
    for step in range(1, cap + 1):
        proposal = weights - rate * gradient_of(weights, X, y)
        current = loss_of(proposal, X, y)
        if not np.isfinite(current) or current > previous:
            rate = rate * shrink
            continue
        weights, previous = proposal, current
        rate = rate * grow
        used.append(rate)
        if current - target < 1e-6:
            return step, float(np.median(used))
    return None, float(np.median(used)) if used else np.nan


for start in [1.0, 0.4, 0.1]:
    steps, typical = bold_driver(ravine, price, start)
    print("bold driver from %-5g -> %s steps, typical rate %.4f" % (start, steps, typical))

**About 6,700 steps from any starting rate, beating the best fixed rate found by hand (7,022 at 0.49).**

It converges to a typical rate of roughly **0.55** - slightly *above* the nominal limit, which it gets
away with because it rejects the steps that make things worse rather than living with them.

**The point of the exercise is not the schedule.** It is that this whole apparatus - two extra
constants, an accept/reject test, and 6,700 steps - is what you need when the problem is badly
conditioned. Standardising the near-duplicate away instead would have made it a 4-step problem. **Adaptive
learning rates are a way to survive a bad loss surface, not a substitute for not having one.**

### E11 · Two budgets, two answers

In [ ]:
def minibatch_descent(X, y, batch_size, learning_rate, epochs, seed=0):
    rng_local = np.random.default_rng(seed)
    weights = np.zeros(X.shape[1])
    updates = 0
    for _ in range(epochs):
        order = rng_local.permutation(len(y))
        for start in range(0, len(y), batch_size):
            chunk = order[start:start + batch_size]
            weights = weights - learning_rate * gradient_of(weights, X[chunk], y[chunk])
            updates += 1
    return weights, updates


def run_updates(X, y, batch_size, learning_rate, total_updates, seed=0):
    rng_local = np.random.default_rng(seed)
    weights = np.zeros(X.shape[1])
    done = 0
    while done < total_updates:
        order = rng_local.permutation(len(y))
        for start in range(0, len(y), batch_size):
            if done >= total_updates:
                break
            chunk = order[start:start + batch_size]
            weights = weights - learning_rate * gradient_of(weights, X[chunk], y[chunk])
            done += 1
    return weights


sizes = [1, 8, 64, 500]
rate = 0.01

by_epoch, by_update = [], []
for size in sizes:
    weights, updates = minibatch_descent(standardised, price, size, rate, 20)
    by_epoch.append({"batch size": size, "updates made": updates,
                     "excess loss": round(loss_of(weights, standardised, price)
                                          - optimal_loss, 6) + 0.0})
    weights = run_updates(standardised, price, size, rate, 2000)
    by_update.append({"batch size": size, "rows read": 2000 * size,
                      "excess loss": round(loss_of(weights, standardised, price)
                                           - optimal_loss, 6) + 0.0})

scaled = []
for size in sizes:
    scaled_rate = min(rate * size, 0.5)          # the "linear scaling rule", capped
    weights, updates = minibatch_descent(standardised, price, size, scaled_rate, 20)
    scaled.append({"batch size": size, "learning rate": scaled_rate, "updates made": updates,
                   "excess loss": round(loss_of(weights, standardised, price)
                                        - optimal_loss, 6) + 0.0})

print("BUDGET: 20 epochs each (every row read 20 times), one rate for all")
print(pd.DataFrame(by_epoch).to_string(index=False), "\n")
print("BUDGET: 2,000 updates each, one rate for all")
print(pd.DataFrame(by_update).to_string(index=False), "\n")
print("BUDGET: 20 epochs each, with the rate scaled to the batch size")
print(pd.DataFrame(scaled).to_string(index=False))

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(12.8, 4.3), sharey=True)
epoch_table, update_table = pd.DataFrame(by_epoch), pd.DataFrame(by_update)
positions = np.arange(len(sizes))

left.bar(positions, epoch_table["excess loss"].clip(lower=1e-4), color="#009E73", width=0.6)
for position, value, updates in zip(positions, epoch_table["excess loss"],
                                    epoch_table["updates made"]):
    left.text(position, max(value, 1e-4) * 1.35, "%d updates" % updates, ha="center", fontsize=9)
left.set_title("Same data read: 20 epochs each", fontsize=11)

right.bar(positions, update_table["excess loss"].clip(lower=1e-4), color="#D55E00", width=0.6)
for position, value, read in zip(positions, update_table["excess loss"],
                                 update_table["rows read"]):
    right.text(position, max(value, 1e-4) * 1.35, "%s rows" % f"{read:,}", ha="center", fontsize=9)
right.set_title("Same updates: 2,000 each", fontsize=11)

for ax in (left, right):
    ax.set_yscale("log")
    ax.set_xticks(positions)
    ax.set_xticklabels(["batch %d" % s for s in sizes])
    ax.set_ylabel("excess loss above the optimum (log)")
plt.tight_layout()
plt.show()

**I would show all three, and the reason is that the first two disagree and the third explains why.**

**By epochs at one shared rate, small batches win** - batch 8 lands at +0.004 while batch 500 is 71,078
out. The annotation says why: batch 1 made 10,000 updates where batch 500 made 20, from the same reading
of the data.

**By updates, the ranking reverses.** 2,000 updates at batch 500 read **a million rows** while 2,000
updates at batch 1 read two thousand. Of course the first is further along; it was allowed five hundred
times as much work.

**The third table shows both of those were measuring something else.** Holding the learning rate fixed
across batch sizes is itself a confound: a gradient from 500 rows is far less noisy than one from a
single row, so it can safely be trusted with a much bigger step. Scaling the rate with the batch size -
the "linear scaling rule" - and rerunning the *same* 20-epoch budget puts batch 500 at the optimum
exactly, in 20 updates, and batch 8 at +0.24.

> **"Which batch size is best" is not a well-posed question.** Batch size and learning rate trade off
> against each other, and comparing batch sizes at a fixed rate compares the pairing, not the size.

**Which leaves a real answer rather than a slogan: what are you short of?** If data reads are the
constraint, larger batches with a proportionally larger rate. If update steps are - a distributed setup
where each update costs a synchronisation, or an online model fed one row at a time - the second table is
the relevant one. And in the common case where neither is binding, mini-batches of 32 to 256 are the
default because they are robust to getting the pairing slightly wrong.

### E12 · Against scikit-learn's `SGDRegressor`

In [ ]:
scaled_only = (raw_features - raw_features.mean(axis=0)) / raw_features.std(axis=0)

settings = [
    ("defaults", dict()),
    ("no penalty, tight tolerance", dict(alpha=0.0, max_iter=50_000, tol=1e-9)),
    ("constant rate 0.01", dict(learning_rate="constant", eta0=0.01,
                                max_iter=50_000, tol=1e-9)),
]
rows = []
for label, options in settings:
    model = SGDRegressor(random_state=0, **options).fit(scaled_only, price)
    weights = np.concatenate([model.intercept_, model.coef_])
    rows.append({"settings": label, "passes made": int(model.n_iter_),
                 "worst coefficient gap": np.abs(weights - best_standardised).max()})

print("closed form:", np.round(best_standardised, 4), "\n")
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: "%.4f" % v))

**`SGDRegressor` gets within 0.035 of the closed form and stops. That is not a bug - it is the design.**

Three sources of disagreement, in order of size:

1. **It stops early.** The default `tol=1e-3` halts when the loss improves by less than a thousandth,
   after 34 passes. Our loop with the tolerance switched off reached 8.5e-14.
2. **It is stochastic.** Updates come from single rows in a shuffled order, so it jitters around the
   minimum rather than settling on it - the wobble in the chapter's mini-batch path.
3. **It penalises by default.** `alpha=1e-4` adds a small ridge term, so the target it is aiming at is
   not quite the least-squares answer. Setting `alpha=0.0` barely changes the gap here, which tells you
   the penalty is *not* the main cause on this data - a useful thing to have measured rather than
   assumed.

The third row shows the default's adaptive schedule earning its keep: forcing a constant rate of 0.01
makes the worst coefficient gap **1.73**, fifty times worse.

**The conclusion is not "sklearn is inaccurate".** It is that `SGDRegressor` exists for problems too
large for the closed form, where 0.035 is far below the noise in the data anyway - here the residual
standard deviation is 18. Using it on 500 rows is using the wrong tool, and `LinearRegression` is exact,
faster and has no settings.

## Interpretation

### E13 · A loss curve that falls steeply then flattens above zero

**Three explanations, all common:**

1. **It converged, and that is the noise floor.** The model has extracted the signal and the remaining
   loss is irreducible - the chapter's fixed route model landed at 3.00 against noise built with sd 3.0.
   This is the good outcome and it looks identical to the others on the plot.
2. **The learning rate is too small and it is still descending**, just too slowly to see on a linear
   axis. The chapter's rate-0.002 run looked flat and was 14,000 above the answer.
3. **The model cannot represent the data** - 05-05's missing squared term, or a missing column. It has
   converged to the best *this model* can do, which is not the best achievable.

**The check that separates them, in order:**

- **Gradient norm.** Near zero rules out (2) immediately. That is one line and it settles half the
  question.
- **If the gradient is near zero, plot the residuals.** Structure means (3); a random cloud means (1).
  This is 05-05 doing the second half of the work.
- **Log scale on the loss axis.** A curve that looks flat linearly is often visibly descending in log,
  and it costs one keyword.

### E14 · Raw features break it, standardised features do not

> "The two columns are on wildly different scales - area runs to 250 and the intercept column is all ones.
> The optimiser has to use one step size for every weight at once, and a step that is sensible for the
> area weight is a thousand times too large for the intercept, or the other way round. So whatever rate
> you pick is simultaneously too big for one direction and far too small for another: it either blows up
> or crawls.
>
> Standardising puts every column on the same footing, so one step size fits all of them, and the same
> code that could not train now trains in four steps."

**What is deliberately left out:** eigenvalues, condition numbers and the loss surface. The teammate
needs to know *what to do* and *why it worked*, and "the columns are on different scales and there is
only one step size" carries both.

## Debugging

### E15 · `nan` after 12 steps

**In this order, because it is the order of decreasing likelihood:**

1. **The learning rate is above the stability limit.** By far the most common cause. Print
   `1 / np.linalg.eigvalsh(X.T @ X / len(X))[-1]` and compare. Overflow to `inf` and then `nan` in about
   a dozen steps is the signature - each step roughly doubles the error.
2. **The features are unscaled.** Usually the same bug wearing different clothes, since it is what makes
   the limit tiny. Check `X.std(axis=0)` for columns differing by orders of magnitude.
3. **Print the loss for the first 12 steps.** Growing steadily means (1). Fine and then suddenly `nan`
   means something else - keep going.
4. **Check the data for `nan` or `inf`.** `np.isfinite(X).all()` and the same for `y`. One bad row
   poisons every weight through the matrix product, immediately.
5. **Check the gradient**, with E8's function. A sign error makes the loop climb rather than descend, and
   climbing a parabola diverges.
6. **Look for a division in the loss** - anything with a denominator that can reach zero. Not applicable
   to squared error, and it is the usual culprit elsewhere.

**Instrument before guessing.** Printing the loss and the gradient norm each step for the first twenty
steps distinguishes all six in one run.

### E16 · Loss falls for 50 epochs then rises steadily

**Cause one: the learning rate is too large for the region it has reached.** As the weights approach the
minimum the gradient shrinks, but if the rate is above the stability limit the *relative* overshoot grows
and the loop starts climbing back out. Steady, smooth growth in the training loss is the signature.

**Cause two: it is the validation loss, and the model is overfitting.** Training loss keeps falling while
held-out loss turns around - the classic picture, and the subject of 05-07 and 05-08.

**How to tell them apart in one look: plot both curves.**

- Both rising -> optimisation. Halve the rate.
- Training falling, validation rising -> overfitting. Stop earlier, or reduce capacity.

**That is why every training loop worth the name logs both**, and it is why "the loss went up" is an
incomplete bug report.

## Exam and interview reasoning

### E17 · "Why gradient descent when there is a closed form?"

> "Because almost nothing else has one. The normal equations work because squared error on a linear model
> is a quadratic, so setting the derivative to zero gives a system you can solve directly. Change the loss
> to logistic, or the model to anything with a nonlinearity, and there is no such solution - but the
> gradient still exists, so descent still works. It is the one method that transfers, which is why the
> same loop trains a neural network."

**"So when would you use the closed form?"**

> "When it is available and the problem is small enough. It is exact, it has no learning rate, no
> stopping rule and no seed, which removes an entire class of bugs. The cost is inverting a `p x p`
> matrix, so it is fine for hundreds of features and not for hundreds of thousands - and it needs the
> data in memory. Roughly: linear regression on a laptop, closed form; anything else, descent."

**What is being tested** is whether you know *why* the closed form exists rather than treating both as
interchangeable tools. The follow-up checks that you will not reach for an optimiser out of habit on a
problem that has an exact answer.

## Transfer to a different situation

### E18 · Forty million rows that do not fit in memory

**What does not change - and this is most of the chapter:**

- The gradient formula is identical. It is a sum over rows, and a sum can be accumulated in pieces.
- Scaling is still the single highest-value preprocessing step, and now it matters more, because you
  cannot afford millions of wasted steps.
- The stability bound still governs the learning rate, and momentum is still worth what the conditioning
  is bad.
- The gradient norm is still the right convergence test.

**What changes:**

- **Full batch is no longer available**, so the choice between batch sizes is made for you. Mini-batch is
  not an optimisation any more, it is the only option.
- **Standardising needs one streaming pass** to accumulate counts, sums and sums of squares - and it must
  use training rows only, which is 04-06's rule under a new constraint.
- **The epoch stops being the natural unit.** With 40 million rows a single pass may be more training
  than you need, so progress is measured in updates or wall-clock time and evaluated on a held-out
  sample periodically.
- **Shuffling becomes a real engineering problem.** If the file is sorted by date, sequential mini-batches
  are all from one period, and each gradient is biased - 05-05's row-order plot, now with consequences.
  Shuffle the file once on disk.
- **The convergence test needs a subsample.** Computing the exact loss requires a full pass, so track it
  on a fixed held-out chunk instead.

**The judgement being tested:** scale changes the engineering and leaves the mathematics alone. Someone
who says "everything is different at scale" has not understood that the gradient is a sum.

## Explain it to someone non-technical

### E19 · What gradient descent is doing

> Imagine standing on a foggy hillside, trying to reach the bottom of the valley. You cannot see where the
> bottom is, but you can feel which way the ground slopes under your feet. So you take a step downhill,
> feel again, and step again. Do that enough times and you arrive, without ever having seen the valley.
>
> The only real decision is how big a step to take. Too small and you are there all week. Too large and
> you stride straight over the valley and up the opposite slope, and end up higher than you started.

*(93 words.)* The step-size sentence is the part that earns its place: it is the learning rate, and it
makes the failure modes intuitive without a single number.

## Optional challenge

### E20 · Deriving the stability condition, and watching the cliff

For a quadratic loss the gradient is linear in the weights. Writing $w^{*}$ for the minimiser and
$H$ for the constant Hessian, the gradient at $w$ is $H(w - w^{*})$, so one step gives:

$$w_{k+1} - w^{*} = (I - \eta H)\left(w_{k} - w^{*}\right)$$

**The error is multiplied by the matrix $I - \eta H$ at every step.** Decompose it along the
eigenvectors of $H$: along the direction with eigenvalue $\lambda$, the error is multiplied by
$1 - \eta\lambda$ each time. That shrinks only if:

$$\left|1 - \eta\lambda\right| < 1 \quad \Longleftrightarrow \quad 0 < \eta < \frac{2}{\lambda}$$

Every direction must shrink, so the binding constraint is the largest eigenvalue:

$$\eta < \frac{2}{\lambda_{\max}(H)}$$

and with $H = \frac{2}{n}X^{\top}X$ that is $\eta < 1 / \lambda_{\max}\!\left(\frac{1}{n}X^{\top}X\right)$.

**The derivation also hands you the speed.** The *slowest* direction shrinks by $1 - \eta\lambda_{\min}$
per step, so the step count is governed by the ratio $\lambda_{\max}/\lambda_{\min}$ - the condition
number. That is E21.

In [ ]:
def distance_history(X, y, rate, steps):
    optimum = np.linalg.lstsq(X, y, rcond=None)[0]
    weights = np.zeros(X.shape[1])
    out = []
    for _ in range(steps):
        weights = weights - rate * gradient_of(weights, X, y)
        out.append(np.linalg.norm(weights - optimum))
    return np.array(out)


eigen = np.linalg.eigvalsh(standardised.T @ standardised / n_houses)
boundary = 1 / eigen[-1]
slowest_factor = abs(1 - 0.9 * boundary * 2 * eigen[0])

print("the boundary is %.6f" % boundary)
print("just inside  (0.99x): predicted per-step shrink in the slowest direction %.6f"
      % abs(1 - 0.99 * boundary * 2 * eigen[0]))
print("just outside (1.01x): predicted per-step growth in the steepest direction %.6f"
      % abs(1 - 1.01 * boundary * 2 * eigen[-1]))

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(13, 4.6))

trails = {}
for fraction, colour in [(0.90, "#009E73"), (0.99, "#0072B2"),
                         (1.01, "#E69F00"), (1.10, "#D55E00")]:
    trails[fraction] = (distance_history(standardised, price, fraction * boundary, 600), colour)

for fraction, (trail, colour) in trails.items():
    left.plot(np.arange(1, 61), trail[:60], color=colour, linewidth=2.2,
              label="%.2f x the boundary" % fraction)
left.set_yscale("log")
left.set_ylim(1e-2, 1e4)
left.set_xlabel("step")
left.set_ylabel("distance to the optimum (log scale)")
left.set_title("The first 60 steps: 1.01 is falling too", fontsize=11)
left.legend(fontsize=8.5, loc="lower left")

for fraction, (trail, colour) in trails.items():
    right.plot(np.arange(1, 601), trail, color=colour, linewidth=2.2,
               label="%.2f x the boundary" % fraction)
right.axvline(60, color="#000000", linewidth=1.4, linestyle=":")
right.text(72, 1e30, "the window on the left", fontsize=9, color="#444444")
right.set_yscale("log")
right.set_xlabel("step")
right.set_ylabel("distance to the optimum (log scale)")
right.set_title("All 600: the 1.01 run turns around", fontsize=11)

plt.tight_layout()
plt.show()

**The two curves inside the bound fall in straight lines on a log axis**, which is exactly the geometric
decay $(1 - \eta\lambda)^{k}$ predicts - the derivation is visible in the picture. The 1.10 run rises in
a straight line for the same reason, reaching 1.65e+49 by step 600.

**The 1.01 curve is the one to study, because it lies.** On the left panel it falls from 375.5 at step 1
to a low of **110.8 at step 32**, then creeps back to 138.9 by step 50 - a shape any practitioner would
read as "converging, nearly there". Stop and report there and you would be reporting a model that is in
fact diverging. It is not: the direction with
the largest eigenvalue is growing by 2% per step while every other direction still shrinks, and the total
distance falls until the growing part takes over. By step 600 it is at **7.4e+06**.

**So the boundary is sharp in theory and slow to reveal itself in practice.** At 0.99 the run converges,
at 1.01 it eventually explodes, the two are 2% apart in learning rate, and a fifty-step diagnostic cannot
tell them apart. That is why "lower the learning rate until it stops exploding" is poor practice twice
over: it leaves you beside a cliff, and a short run may not show you that you are already over it.

### E21 · Steps grow with the condition number

In [ ]:
def design_with_condition(target, rows=400, columns=3, seed=1):
    maker = np.random.default_rng(seed)
    basis, _, rotation = np.linalg.svd(maker.normal(size=(rows, columns)),
                                       full_matrices=False)
    stretched = basis @ np.diag(np.geomspace(1, np.sqrt(target), columns)) @ rotation
    stretched = stretched / stretched.std(axis=0)
    return np.column_stack([np.ones(rows), stretched])


rows = []
for target in [1, 10, 100, 1000]:
    X = design_with_condition(target)
    eigen_here = np.linalg.eigvalsh(X.T @ X / len(X))
    condition = eigen_here[-1] / eigen_here[0]
    truth = np.array([2.0, 1.0, -1.0, 0.5])
    y_here = X @ truth + np.random.default_rng(3).normal(0, 0.5, len(X))

    best_here = np.linalg.lstsq(X, y_here, rcond=None)[0]
    floor = float(((X @ best_here - y_here) ** 2).mean())
    rate = 0.9 / eigen_here[-1]
    weights, count = np.zeros(4), None
    for step in range(1, 400_001):
        weights = weights - rate * (2.0 / len(X) * X.T @ (X @ weights - y_here))
        if float(((X @ weights - y_here) ** 2).mean()) - floor < 1e-9:
            count = step
            break
    rows.append({"condition number": condition, "steps": count,
                 "steps / condition": count / condition})

print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: "%.2f" % v))

**The last column is the finding: 5.57, 4.98, 4.55 across condition numbers spanning two orders of
magnitude.** Steps are proportional to the condition number, with a constant of roughly five for this
tolerance.

**The first row is the honest exception.** At condition 1.46 the ratio reads 32, not 5, because the run
takes 47 steps regardless - at that point the cost is dominated by getting close enough to satisfy a
tolerance of 1e-9 at all, not by the shape of the bowl. The proportionality is an asymptotic statement
about badly conditioned problems, and it is exactly there that it matters.

**Why this is the single most useful number in the chapter.** It converts a property you can compute in
one line, before training anything, into a prediction of how long training will take - and it says the
payoff from improving conditioning is *linear*. Halving the condition number halves the work. That is
what the chapter's 365,727 to 1.09 was buying, and it is why standardising turned 1,776,054 steps into 4.